# Evaluate similarity suggestions

In [1]:
%load_ext autoreload

In [2]:
import warnings
from os.path import join
from IPython.display import display, Markdown, display_html

import scanpy as sc
import pandas as pd
import numpy as np

In [3]:
%autoreload
from datasim.dataset_ot import DatasetMapping

## Load and preprocess data

In [4]:
DATA_PATH = "/vol/data/dataset-similarity"

In [5]:
adata_query = sc.read_h5ad(join(DATA_PATH, "01ad3cd7-3929-4654-84c0-6db05bd5fd59_processed.h5ad"))
adata_ref = sc.read_h5ad(join(DATA_PATH, "b0e547f0-462b-4f81-b31b-5b0a5d96f537_processed.h5ad"))

In [6]:
adata_query.var.set_index("gene_names", inplace=True)
adata_ref.var.set_index("gene_names", inplace=True)

In [7]:
sc.pp.normalize_total(adata_query)
sc.pp.log1p(adata_query)
sc.pp.highly_variable_genes(adata_query, n_top_genes=2000, subset=True)

sc.pp.normalize_total(adata_ref)
sc.pp.log1p(adata_ref)
sc.pp.highly_variable_genes(adata_ref, n_top_genes=2000, subset=True)

/vol/data/miniconda3/envs/similarity/lib/python3.10/site-packages/scanpy/preprocessing/_highly_variable_genes.py:226: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  disp_grouped = df.groupby("mean_bin")["dispersions"]
/vol/data/miniconda3/envs/similarity/lib/python3.10/site-packages/scanpy/preprocessing/_highly_variable_genes.py:226: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  disp_grouped = df.groupby("mean_bin")["dispersions"]


In [8]:
adata_query

AnnData object with n_obs × n_vars = 600929 × 2000
    obs: 'assay', 'cell_type', 'development_stage', 'disease', 'donor_id', 'is_primary_data', 'sex', 'suspension_type', 'tissue', 'cell_type_author'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'log1p', 'hvg'

In [9]:
adata_ref

AnnData object with n_obs × n_vars = 1058909 × 2000
    obs: 'assay', 'cell_type', 'development_stage', 'disease', 'donor_id', 'is_primary_data', 'sex', 'suspension_type', 'tissue', 'cell_type_author'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'log1p', 'hvg'

In [10]:
cluster_mapping = pd.read_parquet(join(DATA_PATH, "model-output/cluster_mapping.parquet"))
cluster_distance = pd.read_parquet(join(DATA_PATH, "model-output/cluster_distance.parquet"))

In [11]:
def extract_ontology_mapping(adata):
    return (
        adata.obs[["cell_type_author", "cell_type"]]
        .drop_duplicates()
        .set_index("cell_type_author")["cell_type"]
        .to_dict()
    )


ontology_mapping_query = extract_ontology_mapping(adata_query)
ontology_mapping_ref = extract_ontology_mapping(adata_ref)

## Select most similar clusters

In [12]:
top_n_labels = DatasetMapping.select_most_similar_clusters(cluster_mapping, cluster_distance, n_top=4)

#### Author provided cluster labels

In [13]:
for i, (k, v) in enumerate(top_n_labels.items()):
    display(Markdown(f"*{i+1}*: **{k}**: {v}"))

*1*: **B_Mem**: ['IGHMhi_memory_B', 'B', 'IGHMlo_memory_B']

*2*: **B_Mem_Prolif_Early**: ['IGHMlo_memory_B', 'IGHMhi_memory_B']

*3*: **B_Mem_Prolif_Late**: ['IGHMlo_memory_B', 'IGHMhi_memory_B']

*4*: **B_Naive**: ['naive_B']

*5*: **B_Naive_Pool3**: []

...

*36*: **cM_Act_1006**: []

*37*: **cM_IFN_1006**: []

*38*: **ncM**: ['CD16+_Monocyte']

*39*: **ncM_1006**: ['CD16+_Monocyte']

*40*: **pDC**: ['pDC', 'DC']

#### Ontology mapped cluster labels

In [14]:
for i, (k, v) in enumerate(top_n_labels.items()):
    display(Markdown(f"*{i+1}*: **{ontology_mapping_query[k]}**: {[ontology_mapping_ref[elem] for elem in v]}"))

*1*: **B cell**: ['memory B cell', 'B cell', 'memory B cell']

*2*: **B cell**: ['memory B cell', 'memory B cell']

*3*: **B cell**: ['memory B cell', 'memory B cell']

*4*: **B cell**: ['naive B cell']

*5*: **B cell**: []

...

*36*: **classical monocyte**: []

*37*: **classical monocyte**: []

*38*: **non-classical monocyte**: ['CD14-low, CD16-positive monocyte']

*39*: **non-classical monocyte**: ['CD14-low, CD16-positive monocyte']

*40*: **plasmacytoid dendritic cell**: ['plasmacytoid dendritic cell', 'dendritic cell']

## Evaluate cluster similarity

In [15]:
%autoreload
from datasim.utils import get_differentially_expressed_genes

In [16]:
def highly_expressed_genes(adata, n_genes):
    highly_expressed = {}
    
    for cluster in adata.obs["cell_type_author"].unique():
        avg_expression = np.array(
            adata[adata.obs["cell_type_author"] == cluster].X.mean(axis=0)
        ).flatten()
        
        highly_expressed_genes_idxs = np.argsort(-avg_expression)[:n_genes]
        
        highly_expressed[cluster] = {
            "gene": adata.var.index[highly_expressed_genes_idxs].tolist(),
            "average_expression": avg_expression[highly_expressed_genes_idxs]
        }

    return highly_expressed


In [17]:
METHOD = "wilcoxon"
P_VAL_THRESHOLD = 0.01

# ignore warnings here as scanpy.tl.rank_genes_groups throws a lot of warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    de_genes_query = get_differentially_expressed_genes(
        adata_query, 
        "cell_type_author", 
        n_genes=15,
        method=METHOD,
        p_value_threshold=P_VAL_THRESHOLD
    )
    de_genes_ref = get_differentially_expressed_genes(
        adata_ref, 
        "cell_type_author", 
        n_genes=15,
        method=METHOD,
        p_value_threshold=P_VAL_THRESHOLD
    )


In [18]:
highly_expressed_query = highly_expressed_genes(adata_query, n_genes=100)
highly_expressed_ref = highly_expressed_genes(adata_ref, n_genes=100)

In [19]:
N_GENES_TO_SHOW = 15


for k, v in top_n_labels.items():
    html = [f"<h1 id='{k}'> <b>{k}</b>: {v} </h1>"]

    if v:
        # Add summary for differentially expressed genes
        html.append("<b><i>Overlap of DE genes:</i></b> <br /><br />")

        def style_overlap(v, props=""):
            top_n_query_genes = de_genes_query[k]["gene"][:N_GENES_TO_SHOW].tolist()
            return "color:green;" if v in top_n_query_genes else "color:red;"

        html += [
            de_genes_query[k]
            .head(N_GENES_TO_SHOW)
            .copy()
            .style
            .format("{:.2f}", subset=["logfoldchg", "score"])
            .format("{:.3f}", subset=["pval_adj"])
            .set_table_attributes("style='display:inline'")
            .set_caption(f"QUERY - {k}")
            ._repr_html_()
        ]
        html += [
            de_genes_ref[gene]
            .head(N_GENES_TO_SHOW)
            .copy()
            .style
            .format("{:.2f}", subset=["logfoldchg", "score"])
            .format("{:.3f}", subset=["pval_adj"])
            .map(style_overlap, subset=["gene"])
            .set_table_attributes("style='display:inline'")
            .set_caption(f"REF - {gene}")
            ._repr_html_()
            for gene in v
        ]
        html.append("<br /><br />")

        # Add summary for highly expressed genes
        html.append("<b><i>Overlap of highly expressed genes:</i></b> <br /><br />")

        def style_overlap(v, propgs=""):
            top_n_query_genes = highly_expressed_query[k]["gene"][:N_GENES_TO_SHOW]
            return "color:green;" if v in top_n_query_genes else "color:red;"

        html += [
            pd.DataFrame(highly_expressed_query[k])
            .head(N_GENES_TO_SHOW)
            .style
            .format("{:.2f}", subset=["average_expression"])
            .set_table_attributes("style='display:inline'")
            .set_caption(f"QUERY - {k}")
            ._repr_html_()
        ]
        html += [
            pd.DataFrame(highly_expressed_ref[gene])
            .head(N_GENES_TO_SHOW)
            .style
            .format("{:.2f}", subset=["average_expression"])
            .map(style_overlap, subset=["gene"])
            .set_table_attributes("style='display:inline'")
            .set_caption(f"REF - {gene}")
            ._repr_html_()
            for gene in v
        ]
    else:
        html.append("<b><i>No matches found</i></b>")

    display_html("".join(html), raw=True)
    with open(join("cluster-evaluation-output", f"{k}.html"), "w") as f:
        f.write("".join(html))

    print("\n")


,gene,logfoldchg,pval_adj,score
0,NXPH4,5.62,0.000,10.47
1,GPM6A,5.04,0.000,8.68
2,FCRL2,4.49,0.000,33.96
3,RASSF6,4.46,0.000,5.72
4,TNFRSF13B,4.45,0.000,40.56
5,CD79A,4.35,0.000,123.80
6,TEAD2,4.32,0.003,3.21
7,MS4A1,4.29,0.000,120.50
8,KLK1,4.17,0.000,12.32
9,CD24,4.15,0.000,52.97


,gene,logfoldchg,pval_adj,score
0,COCH,6.91,0.000,40.83
1,SSPN,6.34,0.000,34.20
2,GRAMD1C,5.47,0.000,30.70
3,RASSF6,5.12,0.000,9.41
4,MTARC2,5.06,0.000,9.85
5,BAIAP3,4.90,0.000,13.73
6,SYBU,4.86,0.003,3.23
7,TNFRSF13B,4.70,0.000,48.85
8,CPNE5,4.60,0.000,53.79
9,EBI3,4.51,0.000,10.36


,gene,logfoldchg,pval_adj,score
0,COCH,6.21,0.000,21.36
1,EBI3,6.03,0.000,14.99
2,GNG3,5.58,0.000,5.32
3,GRAMD1C,5.54,0.000,20.41
4,HRK,5.50,0.000,7.65
5,SSPN,5.44,0.000,15.80
6,CPNE5,5.32,0.000,33.69
7,TNFRSF13B,5.05,0.000,29.08
8,RASSF6,5.02,0.000,5.69
9,ACY3,4.82,0.000,3.94


,gene,logfoldchg,pval_adj,score
0,TCL1A,7.85,0.000,240.03
1,SYN3,7.67,0.005,2.97
2,TCL1B,7.62,0.000,8.97
3,KCNG1,7.48,0.000,10.61
4,SLC38A11,6.93,0.000,14.33
5,COL19A1,6.82,0.000,72.52
6,DSP,6.72,0.000,8.59
7,FCER2,6.56,0.000,193.13
8,CORO2B,6.35,0.009,2.82
9,CLEC20A,6.06,0.000,12.59


B_Naive_Pool3 : [] No matches found

...

cM_Act_1006 : [] No matches found

cM_IFN_1006 : [] No matches found

,gene,logfoldchg,pval_adj,score
0,LYPD2,8.18,0.000,43.71
1,VMO1,7.50,0.000,46.23
2,C1QB,6.99,0.000,54.64
3,CDKN1C,6.81,0.000,133.26
4,C1QC,6.79,0.000,39.05
5,C1QA,6.51,0.000,95.22
6,SPIC,6.42,0.005,3.05
7,CKB,6.23,0.000,64.94
8,HES4,5.21,0.000,127.90
9,MS4A7,4.56,0.000,225.82


,gene,logfoldchg,pval_adj,score
0,CXCL10,7.74,0.000,21.43
1,KLK1,6.72,0.000,24.14
2,APOBEC3A,6.69,0.000,36.73
3,SERPING1,6.58,0.000,34.63
4,IFIT1,6.44,0.000,34.88
5,IFI27,5.97,0.000,30.44
6,IDO1,5.74,0.000,4.04
7,CTSL,5.69,0.000,35.23
8,RSAD2,5.61,0.000,31.74
9,IFIT3,5.53,0.000,35.57


,gene,logfoldchg,pval_adj,score
0,SCT,12.80,0.000,56.81
1,LRRC26,12.56,0.000,53.40
2,SHD,11.99,0.000,34.56
3,KRT5,11.48,0.000,12.72
4,CLEC4C,10.52,0.000,63.44
5,LILRA4,10.09,0.000,74.35
6,PTPRS,8.90,0.000,36.39
7,TPM2,8.39,0.000,59.52
8,SERPINF1,8.16,0.000,70.84
9,PTCRA,8.06,0.000,48.54
